# Multi-Agent Reinforcement Learning Tutorial

This notebook demonstrates multi-agent RL using MADDPG (Multi-Agent Deep Deterministic Policy Gradient).

## What is Multi-Agent RL?

Multi-agent RL involves training multiple agents that interact with each other and the environment. Agents can:
- **Cooperate**: Work together towards a common goal
- **Compete**: Compete against each other
- **Mix**: Both cooperate and compete

## MADDPG Algorithm

MADDPG uses:
- **Centralized Training**: Critics see all agents' observations during training
- **Decentralized Execution**: Actors only use local observations during execution
- **Actor-Critic**: Each agent has its own actor and critic networks

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from rl_llm_toolkit.multiagent import MADDPGAgent, CooperativeNavigationEnv

print("✅ Imports successful!")

## Step 1: Create Multi-Agent Environment

We'll use a cooperative navigation task where agents must reach landmarks while avoiding collisions.

In [ ]:
num_agents = 3
num_landmarks = 3

env = CooperativeNavigationEnv(
    num_agents=num_agents,
    num_landmarks=num_landmarks,
    world_size=2.0,
    agent_size=0.05,
    landmark_size=0.1,
)

print(f"Environment created:")
print(f"  Number of agents: {num_agents}")
print(f"  Number of landmarks: {num_landmarks}")
print(f"  Agent observation space: {env.agent_observation_space}")
print(f"  Agent action space: {env.agent_action_space}")

## Step 2: Visualize Initial State

In [ ]:
def visualize_environment(env, title="Environment State"):
    """Visualize agent and landmark positions."""
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Plot world boundaries
    ax.set_xlim(-env.world_size, env.world_size)
    ax.set_ylim(-env.world_size, env.world_size)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    
    # Plot landmarks
    for i, pos in enumerate(env.landmark_positions):
        circle = plt.Circle(pos, env.landmark_size, color='gold', 
                          alpha=0.5, label='Landmark' if i == 0 else '')
        ax.add_patch(circle)
        ax.text(pos[0], pos[1], f'L{i}', ha='center', va='center')
    
    # Plot agents
    colors = ['blue', 'red', 'green', 'purple', 'orange']
    for i, pos in enumerate(env.agent_positions):
        circle = plt.Circle(pos, env.agent_size, color=colors[i % len(colors)],
                          alpha=0.7, label=f'Agent {i}')
        ax.add_patch(circle)
        ax.text(pos[0], pos[1], f'A{i}', ha='center', va='center', 
               color='white', fontweight='bold', fontsize=8)
    
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

# Reset and visualize
observations, _ = env.reset(seed=42)
visualize_environment(env, "Initial State")

## Step 3: Create MADDPG Agent

In [ ]:
agent = MADDPGAgent(
    env=env,
    num_agents=num_agents,
    learning_rate_actor=1e-4,
    learning_rate_critic=1e-3,
    buffer_size=100000,
    batch_size=64,
    gamma=0.95,
    tau=0.01,
    noise_scale=0.1,
    seed=42,
)

print(f"✅ MADDPG agent created with {num_agents} agents")
print(f"  Actor networks: {len(agent.actors)}")
print(f"  Critic networks: {len(agent.critics)}")
print(f"  Replay buffer size: {agent.buffer_size}")

## Step 4: Train MADDPG

Training will take a few minutes. Watch the agents learn to cooperate!

In [ ]:
print("Training MADDPG agents...")
print("This may take a few minutes...\n")

results = agent.train(
    total_timesteps=50000,
    log_interval=5000,
    progress_bar=True,
)

print(f"\n✅ Training complete!")
print(f"Total timesteps: {results['total_timesteps']}")
print(f"Total episodes: {results['episodes']}")

## Step 5: Visualize Training Progress

In [ ]:
stats = agent.get_training_stats()

fig, axes = plt.subplots(1, num_agents, figsize=(15, 4))

for i in range(num_agents):
    key = f"agent_{i}_rewards"
    if key in stats['stats'] and stats['stats'][key]:
        rewards = stats['stats'][key]
        
        # Moving average
        window = 10
        if len(rewards) >= window:
            moving_avg = np.convolve(rewards, np.ones(window)/window, mode='valid')
            axes[i].plot(rewards, alpha=0.3, label='Raw')
            axes[i].plot(range(window-1, len(rewards)), moving_avg, 
                        linewidth=2, label='Moving Avg')
        else:
            axes[i].plot(rewards)
        
        axes[i].set_xlabel('Episode')
        axes[i].set_ylabel('Reward')
        axes[i].set_title(f'Agent {i} Training')
        axes[i].grid(alpha=0.3)
        axes[i].legend()

plt.tight_layout()
plt.show()

## Step 6: Evaluate Trained Agents

In [ ]:
print("Running evaluation episode...\n")

observations, _ = env.reset(seed=123)
episode_rewards = [0.0] * num_agents
done = False
step = 0
max_steps = 100

trajectory = {
    'agent_positions': [env.agent_positions.copy()],
    'landmark_positions': env.landmark_positions.copy(),
    'collisions': [],
    'coverage': [],
}

while not done and step < max_steps:
    # Get actions from all agents
    actions = {}
    for i in range(num_agents):
        obs = observations[f"agent_{i}"]
        action = agent._get_action(i, obs, add_noise=False)
        actions[f"agent_{i}"] = action
    
    # Step environment
    observations, rewards, terminated, truncated, info = env.step(actions)
    
    # Track metrics
    for i in range(num_agents):
        episode_rewards[i] += rewards[f"agent_{i}"]
    
    trajectory['agent_positions'].append(env.agent_positions.copy())
    trajectory['collisions'].append(info['collisions'])
    trajectory['coverage'].append(info['coverage'])
    
    done = any(terminated.values()) or any(truncated.values())
    step += 1
    
    if step % 20 == 0:
        print(f"Step {step}:")
        print(f"  Collisions: {info['collisions']}")
        print(f"  Coverage: {info['coverage']:.2%}")
        print(f"  Rewards: {[f'{r:.2f}' for r in episode_rewards]}")

print(f"\n{'='*60}")
print("Episode Summary")
print(f"{'='*60}")
for i in range(num_agents):
    print(f"Agent {i}: Total Reward = {episode_rewards[i]:.2f}")
print(f"\nFinal Coverage: {info['coverage']:.2%}")
print(f"Total Collisions: {sum(trajectory['collisions'])}")

## Step 7: Visualize Agent Trajectories

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

# Plot world boundaries
ax.set_xlim(-env.world_size, env.world_size)
ax.set_ylim(-env.world_size, env.world_size)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Plot landmarks
for i, pos in enumerate(trajectory['landmark_positions']):
    circle = plt.Circle(pos, env.landmark_size, color='gold', alpha=0.5)
    ax.add_patch(circle)
    ax.text(pos[0], pos[1], f'L{i}', ha='center', va='center', fontweight='bold')

# Plot agent trajectories
colors = ['blue', 'red', 'green', 'purple', 'orange']
for agent_id in range(num_agents):
    positions = np.array([pos[agent_id] for pos in trajectory['agent_positions']])
    
    # Plot trajectory
    ax.plot(positions[:, 0], positions[:, 1], 
           color=colors[agent_id], alpha=0.6, linewidth=2,
           label=f'Agent {agent_id}')
    
    # Plot start position
    ax.scatter(positions[0, 0], positions[0, 1], 
              color=colors[agent_id], s=100, marker='o', 
              edgecolors='black', linewidths=2, zorder=5)
    
    # Plot end position
    ax.scatter(positions[-1, 0], positions[-1, 1], 
              color=colors[agent_id], s=100, marker='s',
              edgecolors='black', linewidths=2, zorder=5)

ax.set_title('Agent Trajectories\n(○ = Start, □ = End)', 
            fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## Step 8: Analyze Cooperation Metrics

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot coverage over time
ax1.plot(trajectory['coverage'], linewidth=2, color='green')
ax1.axhline(y=1.0, color='red', linestyle='--', label='Perfect Coverage')
ax1.set_xlabel('Step')
ax1.set_ylabel('Coverage')
ax1.set_title('Landmark Coverage Over Time')
ax1.set_ylim([0, 1.1])
ax1.grid(alpha=0.3)
ax1.legend()

# Plot collisions over time
ax2.plot(trajectory['collisions'], linewidth=2, color='red')
ax2.set_xlabel('Step')
ax2.set_ylabel('Number of Collisions')
ax2.set_title('Agent Collisions Over Time')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Print statistics
print(f"\nCooperation Metrics:")
print(f"  Average Coverage: {np.mean(trajectory['coverage']):.2%}")
print(f"  Final Coverage: {trajectory['coverage'][-1]:.2%}")
print(f"  Total Collisions: {sum(trajectory['collisions'])}")
print(f"  Average Collisions per Step: {np.mean(trajectory['collisions']):.2f}")

## Step 9: Save Trained Model

In [ ]:
model_path = Path("./models/multiagent/maddpg_navigation.pt")
model_path.parent.mkdir(parents=True, exist_ok=True)

agent.save(model_path)
print(f"✅ Model saved to {model_path}")

## Key Takeaways

1. **Multi-agent RL** enables training agents that interact with each other
2. **MADDPG** uses centralized training with decentralized execution
3. **Cooperation** emerges through shared rewards and communication
4. **Metrics** like coverage and collisions help evaluate cooperation quality
5. Applications include:
   - Robotics swarms
   - Traffic control
   - Game AI
   - Distributed systems

## Next Steps

- Try different numbers of agents and landmarks
- Experiment with competitive scenarios
- Add communication between agents
- Apply to custom multi-agent environments

In [ ]:
print("\n✅ Multi-agent tutorial complete!")